In [1]:
!pip install -q langchain langchain-openai langchain-community pydantic langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
import os
from google.colab import userdata

# ================== PUT YOUR KEYS HERE ==================
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY') or "sk-your-openai-key-here"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('LANGCHAIN_API_KEY') or "lsv2-your-langsmith-key-here"
os.environ["LANGCHAIN_PROJECT"] = "Recruiter-AI-Tool-v1"

print("✅ Environment setup complete!")
print(f"Project: {os.environ['LANGCHAIN_PROJECT']}")

In [ ]:
job_description = """We are hiring a Senior Data Scientist.

Requirements:
- 5+ years of experience in Data Science or Machine Learning
- Strong proficiency in Python, SQL
- Hands-on experience with ML frameworks: Scikit-learn, TensorFlow or PyTorch
- Experience with big data tools: Spark, Hadoop
- Cloud experience (AWS/GCP) is a plus
- Bachelor's or Master's in CS, Statistics, or related field"""

strong_resume = """John Doe - Senior Data Scientist
Experience: 6+ years
• Built production ML models using Python, TensorFlow, Scikit-learn at TechCorp (2020-Present)
• Led Spark pipelines at DataInc (2018-2020)
Skills: Python, SQL, Scikit-learn, TensorFlow, PyTorch, Spark, AWS, Git
Education: MS Data Science, Stanford"""

average_resume = """Jane Smith - Data Analyst
Experience: 3 years
• Performed data analysis and built basic ML models using Python and Scikit-learn
Skills: Python, SQL, Excel, Scikit-learn, Tableau
Education: BSc Statistics"""

weak_resume = """Bob Johnson - Marketing Executive
Experience: 1 year
• Handled marketing campaigns and basic Excel work
Skills: MS Office, Communication, Digital Marketing
Education: BA Marketing"""

print("✅ Sample data loaded")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnablePassthrough
import json

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ LLM initialized")

In [ ]:
extract_prompt = PromptTemplate.from_template(
"""You are an expert resume parser. Extract information **only** from the given resume text.
Do NOT assume or hallucinate any skills, experience, or tools.

Resume:
{resume}

Return JSON only with this structure:
{{
  "skills": ["list of skills"],
  "experience_years": number or null,
  "tools": ["list of tools/frameworks"],
  "education": "degree info or null"
}}"""
)

extract_chain = extract_prompt | llm | JsonOutputParser()

print("✅ Skill Extraction Chain Ready")

In [ ]:
def evaluate_candidate(name, resume_text):
    print(f"\n{'='*70}")
    print(f"🚀 Evaluating: {name.upper()} CANDIDATE")
    print(f"{'='*70}")

    # Step 1: Extract Skills
    candidate_info = extract_chain.invoke({"resume": resume_text})

    # Step 2: Matching & Scoring Prompt
    scoring_prompt = PromptTemplate.from_template(
"""Compare the candidate with the job description.

Job Description:
{job_description}

Candidate Extracted Info:
{candidate_info}

Evaluate fairly and return JSON only:
{{
  "fit_score": number (0-100),
  "strengths": ["list of strengths"],
  "gaps": ["list of gaps"],
  "explanation": "detailed explanation why this score was given"
}}"""
    )

    scoring_chain = scoring_prompt | llm | JsonOutputParser()

    result = scoring_chain.invoke({
        "job_description": job_description,
        "candidate_info": json.dumps(candidate_info, indent=2)
    })

    print(f"✅ Fit Score: **{result['fit_score']}**/100\n")
    print("📝 Explanation:")
    print(result['explanation'])

    return result

print("✅ Pipeline function ready")

In [ ]:
candidates = {
    "Strong": strong_resume,
    "Average": average_resume,
    "Weak": weak_resume
}

results = {}

for name, resume in candidates.items():
    results[name] = evaluate_candidate(name, resume)

print("\n🎉 All evaluations completed!")
print("📊 Check full tracing in LangSmith → https://smith.langchain.com")